## Preamble and sampling

In [1]:
import nltk
import polars as pl
import numpy as np

from collections import Counter
from datasets import load_dataset
from nltk.tokenize import word_tokenize

/home/ubuntu/miniconda/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nltk.download("punkt")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/ubuntu/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /home/ubuntu/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
load_dataset("roneneldan/TinyStories", split="train").to_parquet("tinystories.parquet")
load_dataset("SimpleStories/SimpleStories", split="train").to_parquet("simplestories.parquet")

Creating parquet from Arrow format: 100%|██████████| 32/32 [00:19<00:00,  1.61ba/s]


3142783327

In [4]:
tinystories_df = pl.scan_parquet("tinystories.parquet")
# tinystories_df = pl.scan_parquet("tinystories.parquet").head(50000).collect()

In [5]:
simplestories_df = pl.scan_parquet("simplestories.parquet")
# simplestories_df = pl.scan_parquet("simplestories.parquet").head(50000).collect()

## Length comparison

In [6]:
tinystories_df = tinystories_df.with_columns(
    pl.col("text").str.split(" ").list.len().alias("word_count")
)

In [7]:
%%time
"""
flesch-kincaid approx function in pure polars
"""
tinystories_df = tinystories_df.with_columns([
    # Count words
    pl.col("text").str.split(" ").list.len().alias("word_count"),
    # Count sentences (split on . ! ?)
    pl.col("text").str.count_matches(r"[.!?]+").alias("sentence_count"),
    # Approximate syllables: count vowel groups per word
    pl.col("text").str.count_matches(r"[aeiouAEIOU]+").alias("syllable_count"),
]).with_columns(
    (
        0.39 * (pl.col("word_count") / pl.col("sentence_count").clip(lower_bound=1))
        + 11.8 * (pl.col("syllable_count") / pl.col("word_count").clip(lower_bound=1))
        - 15.59
    ).alias("flesch_kincaid_score")
)

simplestories_df = simplestories_df.with_columns([
    # Count words
    pl.col("story").str.split(" ").list.len().alias("word_count"),
    # Count sentences (split on . ! ?)
    pl.col("story").str.count_matches(r"[.!?]+").alias("sentence_count"),
    # Approximate syllables: count vowel groups per word
    pl.col("story").str.count_matches(r"[aeiouAEIOU]+").alias("syllable_count"),
]).with_columns(
    (
        0.39 * (pl.col("word_count") / pl.col("sentence_count").clip(lower_bound=1))
        + 11.8 * (pl.col("syllable_count") / pl.col("word_count").clip(lower_bound=1))
        - 15.59
    ).alias("flesch_kincaid_score")
)

CPU times: user 297 μs, sys: 0 ns, total: 297 μs
Wall time: 302 μs


In [8]:
ts_result = tinystories_df.select(
    word_count_mean = pl.col("word_count").mean(),
    word_count_std = pl.col("word_count").std(),
    flesch_kincaid_mean = pl.col("flesch_kincaid_score").mean(),
    flesch_kincaid_std = pl.col("flesch_kincaid_score").std(),
    
)

ss_result = simplestories_df.select(
    word_count_mean = pl.col("word_count").mean(),
    word_count_std = pl.col("word_count").std(),
    flesch_kincaid_mean = pl.col("flesch_kincaid_score").mean(),
    flesch_kincaid_std = pl.col("flesch_kincaid_score").std(),
    
)

In [9]:
%%time
print("Tiny stories")
ts_result.collect()

Tiny stories
CPU times: user 54.1 s, sys: 2.79 s, total: 56.9 s
Wall time: 8.06 s


word_count_mean,word_count_std,flesch_kincaid_mean,flesch_kincaid_std
f64,f64,f64,f64
171.832831,77.416249,4.108448,1.513775


In [10]:
%%time
print("Simple stories")
ss_result.collect()

Simple stories
CPU times: user 1min 14s, sys: 874 ms, total: 1min 14s
Wall time: 10.4 s


word_count_mean,word_count_std,flesch_kincaid_mean,flesch_kincaid_std
f64,f64,f64,f64
222.87192,102.668153,4.936471,1.348981


## Compression ratio and Self-BLEU homogenization score

In [12]:
# random subsample from each 
from diversity import (
    compression_ratio,
    homogenization_score,
    ngram_diversity_score,
)

[nltk_data] Downloading package punkt_tab to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [36]:
tinystories_df_sample = tinystories_df.collect().sample(1000, seed=1)
simplestories_df_sample = simplestories_df.collect().sample(1000, seed=1)

In [37]:
ts_cr = compression_ratio(tinystories_df_sample["text"], algorithm='gzip')
print(f"Tiny stories Compression Ratio: {ts_cr:.4f}")

ss_cr = compression_ratio(simplestories_df_sample["story"], algorithm='gzip')
print(f"Simple stories Compression Ratio: {ss_cr:.4f}")

# tr_hs = homogenization_score(tinystories_df_sample["text"], measure='bleu')
# print(f"Homogenization (Self-BLEU): {tr_hs:.4f}")

Tiny stories Compression Ratio: 3.2000
Simple stories Compression Ratio: 2.9590


In [39]:
import numpy as np
from scipy import stats
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def homogenization_score_with_ci(texts, measure='bleu', confidence=0.999):
    """
    Calculate homogenization score with confidence interval.
    
    Args:
        texts: List or Series of text strings
        measure: 'bleu' 
        confidence: Confidence level (0.999 for 99.9%)
    
    Returns:
        dict with 'mean', 'ci_lower', 'ci_upper', 'scores'
    """
    # Convert to list if needed
    if hasattr(texts, 'tolist'):
        texts = texts.tolist()
    
    # Tokenize texts
    tokenized_texts = [text.split() for text in texts]
    tokenized_texts = [t for t in tokenized_texts if len(t) >= 3]
    
    if len(tokenized_texts) < 2:
        return {'mean': 0.0, 'ci_lower': 0.0, 'ci_upper': 0.0, 'scores': []}
    
    # Calculate Self-BLEU scores
    bleu_scores = []
    smoothing = SmoothingFunction().method1
    
    for i, hypothesis in enumerate(tokenized_texts):
        # Use all other texts as references
        references = [tokenized_texts[j] for j in range(len(tokenized_texts)) if j != i]
        
        # Calculate BLEU score
        score = sentence_bleu(
            references, 
            hypothesis,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoothing
        )
        bleu_scores.append(score)
    
    # Calculate statistics
    mean_score = np.mean(bleu_scores)
    std_score = np.std(bleu_scores, ddof=1)  # Sample standard deviation
    n = len(bleu_scores)
    
    # Calculate confidence interval
    # Using normal distribution assumption (as stated in figure caption)
    # Z-score for 99.9% confidence interval
    alpha = 1 - confidence
    z_score = stats.norm.ppf(1 - alpha/2)
    
    # Standard error
    se = std_score / np.sqrt(n)
    
    # Confidence interval
    ci_lower = mean_score - z_score * se
    ci_upper = mean_score + z_score * se
    
    return {
        'mean': mean_score,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'std': std_score,
        'n': n,
        'scores': bleu_scores
    }

# Calculate scores with confidence intervals
ts_results = homogenization_score_with_ci(tinystories_df_sample["text"], confidence=0.999)
print(f"Tiny stories homogenization score: {ts_results['mean']:.4f}")
print(f"99.9% CI: [{ts_results['ci_lower']:.4f}, {ts_results['ci_upper']:.4f}]")

ss_results = homogenization_score_with_ci(simplestories_df_sample["story"], confidence=0.999)
print(f"Simple stories homogenization score: {ss_results['mean']:.4f}")
print(f"99.9% CI: [{ss_results['ci_lower']:.4f}, {ss_results['ci_upper']:.4f}]")

Tiny stories homogenization score: 0.4504
99.9% CI: [0.4421, 0.4588]
Simple stories homogenization score: 0.3802
99.9% CI: [0.3729, 0.3875]
